# Phase kickback

When a control qubit in $|{+}\rangle$ targets an eigenvector $|u\rangle$
of a unitary $U$ with eigenvalue $e^{i\phi}$, the controlled-$U$ maps:

$$
|{+}\rangle|u\rangle \;\longrightarrow\; \bigl(\cos\tfrac{\phi}{2}|0\rangle + \sin\tfrac{\phi}{2}|1\rangle\bigr)|u\rangle
$$

The eigenphase $\phi$ kicks back as a rotation on the control.

This demo uses a Toffoli oracle (marks $|11\rangle$) with the ancilla
in $|{-}\rangle$. We show the statevector, explain why single-qubit
measurement gives 50/50, and demonstrate that the Grover diffusion step
converts the invisible phase into a visible amplitude.

In [ ]:
import math
import qiskit as qk
import qiskit_aer as qka

## Statevector analysis

After the Toffoli oracle, the marked state $|11\rangle$ picks up a
minus sign. The ancilla stays in $|{-}\rangle$.

In [ ]:
qc = qk.QuantumCircuit(3)
qc.h(0)
qc.h(1)
qc.x(2)
qc.h(2)
qc.ccx(0, 1, 2)
sv = qk.quantum_info.Statevector.from_instruction(qc)
for state in sorted(sv.probabilities_dict().keys()):
    amp = sv.data[int(state[::-1], 2)]
    p = sv.probabilities_dict()[state]
    sign = "+" if amp.real >= 0 else "-"
    print(f"|{state}>  {sign}{abs(amp):.4f}  p={p:.4f}")

## Why single-qubit measurement gives 50/50

The phase kickback lives in the **two-qubit amplitudes**, not in any
single-qubit reduced state. Tracing out the ancilla gives $I/4$
(maximally mixed), so every qubit individually measures 50/50.

In [ ]:
qc_m = qk.QuantumCircuit(3, 3)
qc_m.h(0)
qc_m.h(1)
qc_m.x(2)
qc_m.h(2)
qc_m.ccx(0, 1, 2)
qc_m.measure([0, 1, 2], [0, 1, 2])
print(qc_m.draw())

sim = qka.AerSimulator()
counts = sim.run(qk.transpile(qc_m, sim), shots=4096).result().get_counts()
for bits, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  |{bits}>  {n:4d}")

## Grover: diffusion makes the phase visible

The Grover diffusion step ($2|s\rangle\langle s| - I$) converts the
invisible phase into a visible amplitude. After one iterate, $|11\rangle$
is amplified.

In [ ]:
qc_g = qk.QuantumCircuit(3, 2)
qc_g.h(0)
qc_g.h(1)
qc_g.x(2)
qc_g.h(2)
qc_g.ccx(0, 1, 2)
# diffusion on q0, q1
qc_g.h(0)
qc_g.h(1)
qc_g.x(0)
qc_g.x(1)
qc_g.h(1)
qc_g.cx(0, 1)
qc_g.h(1)
qc_g.x(0)
qc_g.x(1)
qc_g.h(0)
qc_g.h(1)
qc_g.measure([0, 1], [0, 1])
print(qc_g.draw())

counts_g = sim.run(qk.transpile(qc_g, sim), shots=4096).result().get_counts()
for bits, n in sorted(counts_g.items(), key=lambda kv: -kv[1]):
    flag = "  <-- marked" if bits == "11" else ""
    print(f"  |{bits}>  {n:4d}{flag}")

After one Grover iterate the marked state $|11\rangle$ is amplified.
The diffusion step converts the phase kickback (invisible in
measurement) into an amplitude (visible).